In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

# -------------------------------------------------
# 1. Cargar los datos
# -------------------------------------------------

datos = fetch_openml(
    name="adult",
    version=2,
    as_frame=True
)

X = datos.data
y = datos.target

# Seleccionar una muestra reproducible
muestra = X.sample(n=10000, random_state=42).index
X = X.loc[muestra]
y = y.loc[muestra]

# -------------------------------------------------
# 2. Preparar las variables
# -------------------------------------------------

variables_numericas = X.select_dtypes(
    include=["number"]
).columns.tolist()

variables_categoricas = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

preprocesamiento = ColumnTransformer([
    (
        "numericas",
        SimpleImputer(strategy="median"),
        variables_numericas
    ),
    (
        "categoricas",
        Pipeline([
            ("imputacion",
             SimpleImputer(strategy="most_frequent")),
            ("codificacion",
             OneHotEncoder(handle_unknown="ignore"))
        ]),
        variables_categoricas
    )
])

# -------------------------------------------------
# 3. Separar los datos
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------------------------
# 4. Crear el arbol de decision
# -------------------------------------------------

modelo_arbol = Pipeline([
    ("preprocesamiento", preprocesamiento),
    (
        "clasificador",
        DecisionTreeClassifier(
            criterion="gini",
            max_depth=5,
            min_samples_leaf=10,
            random_state=42
        )
    )
])

# Entrenar el modelo
modelo_arbol.fit(X_train, y_train)

# -------------------------------------------------
# 5. Evaluar el modelo
# -------------------------------------------------

predicciones = modelo_arbol.predict(X_test)

print("Exactitud del arbol:",
      accuracy_score(y_test, predicciones))

print("\nReporte de clasificacion:")
print(classification_report(y_test, predicciones))

# -------------------------------------------------
# 6. Mostrar las reglas aprendidas
# -------------------------------------------------

nombres = modelo_arbol.named_steps[
    "preprocesamiento"
].get_feature_names_out()

arbol = modelo_arbol.named_steps["clasificador"]

reglas = export_text(
    arbol,
    feature_names=list(nombres),
    max_depth=5
)

print("\nReglas del arbol de decision:")
print(reglas)

Exactitud del arbol: 0.8455

Reporte de clasificacion:
              precision    recall  f1-score   support

       <=50K       0.86      0.95      0.90      1532
        >50K       0.76      0.50      0.60       468

    accuracy                           0.85      2000
   macro avg       0.81      0.73      0.75      2000
weighted avg       0.84      0.85      0.83      2000


Reglas del arbol de decision:
|--- categoricas__marital-status_Married-civ-spouse <= 0.50
|   |--- numericas__capital-gain <= 7139.50
|   |   |--- numericas__capital-loss <= 2218.50
|   |   |   |--- numericas__hours-per-week <= 40.50
|   |   |   |   |--- numericas__education-num <= 12.50
|   |   |   |   |   |--- class: <=50K
|   |   |   |   |--- numericas__education-num >  12.50
|   |   |   |   |   |--- class: <=50K
|   |   |   |--- numericas__hours-per-week >  40.50
|   |   |   |   |--- numericas__education-num <= 11.50
|   |   |   |   |   |--- class: <=50K
|   |   |   |   |--- numericas__education-num >  11.